# Stage 6 — Inference on new / out-of-distribution images
Runs the **latest** trained model over a folder of unlabelled images and
writes (1) annotated images with boxes + species, and (2) a CSV of every
detection. No ground truth needed — this is batch prediction / deployment.

Animals are classified; **people are flagged separately** (drawn red, logged)
for security/poaching monitoring and are never sent to the classifier.

In [1]:
import sys; sys.path.insert(0,'/workspace/lib')
import os, glob, csv, time
from PIL import Image, ImageDraw, ImageFont, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES=True
import onnxruntime as ort
from inference_lib import Pipeline
import dataset_lib as D

In [2]:
# --- config: EDIT THESE ---
OOD_DIR   = '/dataset/ood_images'          # folder of images to run on
OUT_DIR   = '/dataset/ood_predictions'     # annotated images + CSV go here
DETECTOR  = '/workspace/cai_uk_mammals_yolo10x_v1.0.0.onnx'  # your YOLO detector ONNX
# classifier: use the latest exported model automatically (highest version)
CLASSIFIER = sorted(glob.glob('/dataset/models/**/*.onnx', recursive=True))[-1]

DET_CONF  = 0.10   # detector confidence threshold (lower = catch more, esp. OOD)
NMS_IOU   = 0.45
PAD       = 0.12
# --------------------------
os.makedirs(OUT_DIR, exist_ok=True)
prov = ort.get_available_providers()
print('providers:', prov)
print('detector :', DETECTOR)
print('classifier:', CLASSIFIER)

providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
detector : /workspace/cai_uk_mammals_yolo10x_v1.0.0.onnx
classifier: /dataset/models/v1.1/deepfaune_uk_v1.1.onnx


In [3]:
# load both models once
pipe = Pipeline(DETECTOR, CLASSIFIER)
print('loaded. classifier knows', len(pipe.classes), 'species at', pipe.size, 'px')

2026-07-02 14:41:29.134857950 [W:onnxruntime:, transformer_memcpy.cc:74 ApplyImpl] 24 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.


loaded. classifier knows 30 species at 518 px


In [4]:
# colours per kind
COL = {'animal': (38,113,51), 'person': (220,40,40), 'vehicle': (120,120,120)}
def draw(frame, results):
    im = frame.copy(); dr = ImageDraw.Draw(im)
    try: font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 18)
    except: font = ImageFont.load_default()
    for r in results:
        x1,y1,x2,y2 = r['box']; c = COL.get(r['kind'], (38,113,51))
        dr.rectangle([x1,y1,x2,y2], outline=c, width=3)
        conf = r['species_conf']
        lab = r['species'] if conf is None else f"{r['species']} {conf:.2f}"
        tb = dr.textbbox((0,0), lab, font=font); tw,th = tb[2]-tb[0], tb[3]-tb[1]
        dr.rectangle([x1, y1-th-6, x1+tw+8, y1], fill=c)
        dr.text((x1+4, y1-th-5), lab, fill=(255,255,255), font=font)
    return im

In [5]:
# run over the folder
exts = ('*.jpg','*.jpeg','*.png','*.JPG','*.JPEG','*.PNG')
paths = []
for e in exts: paths += glob.glob(os.path.join(OOD_DIR, e))
paths = sorted(set(paths))
print(f'{len(paths)} images in {OOD_DIR}')

csv_path = os.path.join(OUT_DIR, 'predictions.csv')
person_path = os.path.join(OUT_DIR, 'person_detections.csv')
n_det = n_person = n_empty = 0
t0 = time.time()
with open(csv_path,'w',newline='') as f, open(person_path,'w',newline='') as pf:
    w = csv.writer(f); pw = csv.writer(pf)
    w.writerow(['image','kind','species','species_conf','detector_class','detector_conf','x1','y1','x2','y2'])
    pw.writerow(['image','detector_conf','x1','y1','x2','y2'])
    for i,p in enumerate(paths,1):
        try:
            frame, results = pipe.infer(p, det_conf=DET_CONF, nms_iou=NMS_IOU, pad=PAD)
        except Exception as ex:
            print('  skip', os.path.basename(p), ex); continue
        if not results: n_empty += 1
        for r in results:
            x1,y1,x2,y2 = r['box']
            w.writerow([os.path.basename(p), r['kind'], r['species'], r['species_conf'],
                        r['detector_class'], r['detector_conf'], x1,y1,x2,y2])
            n_det += 1
            if r['kind']=='person':
                pw.writerow([os.path.basename(p), r['detector_conf'], x1,y1,x2,y2]); n_person += 1
        # save annotated image
        draw(frame, results).save(os.path.join(OUT_DIR, os.path.basename(p)))
        if i % 25 == 0 or i == len(paths):
            print(f'  {i}/{len(paths)} | {n_det} detections | {n_person} person | {n_empty} empty')
print(f'\ndone in {time.time()-t0:.0f}s')
print('annotated images + predictions.csv in', OUT_DIR)
if n_person: print(f'** {n_person} PERSON detection(s) logged to person_detections.csv **')

269 images in /dataset/ood_images
  25/269 | 28 detections | 0 person | 0 empty
  50/269 | 55 detections | 0 person | 0 empty
  75/269 | 82 detections | 0 person | 0 empty
  100/269 | 110 detections | 0 person | 0 empty
  125/269 | 138 detections | 0 person | 0 empty
  150/269 | 164 detections | 0 person | 0 empty
  175/269 | 208 detections | 9 person | 0 empty
  200/269 | 259 detections | 39 person | 0 empty
  225/269 | 290 detections | 43 person | 0 empty
  250/269 | 319 detections | 43 person | 0 empty
  269/269 | 343 detections | 47 person | 0 empty

done in 32s
annotated images + predictions.csv in /dataset/ood_predictions
** 47 PERSON detection(s) logged to person_detections.csv **


In [6]:
# quick summary: species counts across the folder
from collections import Counter
counts = Counter()
with open(csv_path) as f:
    import csv as _c
    for row in _c.DictReader(f):
        counts[row['species']] += 1
print('predicted detections by species:')
for sp,n in counts.most_common():
    print(f'  {n:5d}  {sp}')

predicted detections by species:
     57  MelesMeles
     55  SciurusVulgaris
     50  Person
     44  SciurusCarolinensis
     30  OryctolagusCuniculus
     19  PhasianusColchicus
     18  ColumbaPalumbus
     18  VulpesVulpes
     15  CalibrationPole
     14  OvisAries
     12  EquusCaballus
      6  CanisFamiliaris
      3  FelisCatus
      1  ErinaceusEuropaeus
      1  NumeniusArquata


Outputs in your `OUT_DIR`:
- annotated images (green=animal, red=person, grey=vehicle)
- `predictions.csv` — every detection with species + confidence + box
- `person_detections.csv` — people only, for security review

**OOD tip:** if the model misses animals (many 'empty'), lower `DET_CONF`
(e.g. 0.05). On genuinely out-of-distribution imagery the *detector* is
usually the bottleneck, not the classifier — low-confidence detections are
where new sites/cameras differ most from the training data.